# Универсальный pipeline анализа публикаций arXiv по физике

Код предназначен для автоматического формирования корпуса научных публикаций arXiv и сопоставления терминов, выделенных методами TF-IDF и KeyBERT.

## 1. Установка и импорт библиотек

В проекте используются библиотеки для работы с таблицами, HTTP-запросами, HTML и PDF, а также инструменты машинного обучения и обработки естественного языка.

Ячейку установки достаточно выполнить один раз в рабочем окружении. При последующих запусках notebook ее можно пропустить, если все библиотеки уже установлены.


In [61]:
%pip install -q pandas requests beautifulsoup4 pypdf scikit-learn keybert sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [62]:
from pathlib import Path
from io import BytesIO

import json
import random
import re
import time

import pandas as pd
import requests

from bs4 import BeautifulSoup
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from keybert import KeyBERT


DATA_DIR = Path("pipeline_data")
DATA_DIR.mkdir(exist_ok=True)

RANDOM_SEED = 42

random.seed(RANDOM_SEED)

print("Рабочая папка:", DATA_DIR.resolve())

Рабочая папка: C:\Users\user\Desktop\quantum_physics\pipeline_data


## 2. Получение списка публикаций arXiv

Функция `fetch_month_from_arxiv()` получает список публикаций выбранной категории за конкретный месяц. Для каждой публикации сохраняются arXiv ID, название, тематические категории и месяц публикации.

Используется страница списка arXiv с ограничением до 2000 записей. Если получено ровно 2000 публикаций, программа выводит предупреждение о том, что список за месяц потенциально может быть неполным.


In [63]:
def fetch_month_from_arxiv(category, year, month, retries=4):

    month_label = f"{year}-{month:02d}"

    url = (
        f"https://arxiv.org/list/"
        f"{category}/{month_label}?show=2000"
    )

    headers = {
        "User-Agent":
        "Mozilla/5.0 (academic research; arXiv physics study)"
    }

    for attempt in range(1, retries + 1):

        try:
            response = requests.get(
                url,
                headers=headers,
                timeout=120,
            )

            print(
                f"{category} | {month_label}: "
                f"HTTP {response.status_code}"
            )

            if response.status_code == 200:

                soup = BeautifulSoup(
                    response.text,
                    "html.parser",
                )

                rows = []

                for dt, dd in zip(
                    soup.find_all("dt"),
                    soup.find_all("dd"),
                ):

                    abs_link = dt.find(
                        "a",
                        href=re.compile(r"^/abs/"),
                    )

                    if abs_link is None:
                        continue

                    arxiv_id = abs_link["href"].replace(
                        "/abs/",
                        "",
                    )

                    arxiv_id = re.sub(
                        r"v\d+$",
                        "",
                        arxiv_id,
                    )

                    title_div = dd.find(
                        "div",
                        class_="list-title",
                    )

                    subjects_div = dd.find(
                        "div",
                        class_="list-subjects",
                    )

                    if title_div:
                        title = title_div.get_text(
                            " ",
                            strip=True,
                        )
                    else:
                        title = ""

                    if subjects_div:
                        subjects = subjects_div.get_text(
                            " ",
                            strip=True,
                        )
                    else:
                        subjects = ""

                    title = re.sub(
                        r"^Title:\s*",
                        "",
                        title,
                    )

                    subjects = re.sub(
                        r"^Subjects:\s*",
                        "",
                        subjects,
                    )

                    rows.append({
                        "arxiv_id": arxiv_id,
                        "title": title,
                        "subjects": subjects,
                        "month": month_label,
                        "category": category,
                    })

                    # Для quant-ph оставляем публикации,
                    # в предметных категориях которых явно указана Quantum Physics.
                    # Это сохраняет тот же принцип формирования корпуса,
                    # который использовался в первоначальном исследовании.
                    if category == "quant-ph":
                        rows = [
                            article
                            for article in rows
                            if "Quantum Physics (quant-ph)" in article["subjects"]
                        ]

                # Удаляем возможные повторяющиеся ID
                rows = list({
                    article["arxiv_id"]: article
                    for article in rows
                }.values())

                if rows:

                    print(
                        f"{category} | {month_label}: "
                        f"найдено {len(rows)} публикаций"
                    )

                    if len(rows) >= 2000:
                        print(
                            "ВНИМАНИЕ: достигнут лимит 2000 публикаций. "
                            "Список за этот месяц может быть неполным."
                        )

                    return rows

        except requests.RequestException as error:

            print(
                "Ошибка сети:",
                error,
            )

        if attempt < retries:
            time.sleep(5 * attempt)

    raise RuntimeError(
        f"Не удалось получить {category} "
        f"за {month_label}"
    )


## 3. Формирование корпуса

Корпус состоит из двух групп одинакового размера.

- `RECENT` - публикации за выбранный последний месяц
- `PREVIOUS` - публикации за предыдущие 24 месяца

Для основной работы используется по 50 публикаций в каждой группе. Старые публикации распределяются по двухлетнему периоду максимально равномерно. Фиксированный `RANDOM_SEED` делает случайный выбор воспроизводимым при неизменном исходном списке arXiv.


In [64]:
def build_corpus(
    category,
    recent_year,
    recent_month,
    n_per_group=50,
):

    # Фиксируем генератор случайных чисел,
    # чтобы выборка была воспроизводимой
    random.seed(RANDOM_SEED)

    recent_period = pd.Period(
        f"{recent_year}-{recent_month:02d}",
        freq="M",
    )

    # Формируем период из 24 месяцев,
    # непосредственно предшествующих анализируемому месяцу
    first_previous_month = recent_period - 24
    last_previous_month = recent_period - 1

    months = pd.period_range(
        first_previous_month,
        last_previous_month,
        freq="M",
    )

    previous_rows = []

    # Для основного эксперимента с 50 статьями
    # берём по 2 публикации из каждого месяца.
    # Чтобы получить ровно 50 статей,
    # в первом и последнем месяце берём по 3.
    if n_per_group == 50:

        articles_per_month = [2] * len(months)
        articles_per_month[0] = 3
        articles_per_month[-1] = 3

    else:

        # Для небольших технических тестов
        # распределяем нужное количество статей
        # по всему рассматриваемому периоду
        base_number = n_per_group // len(months)
        extra_number = n_per_group % len(months)

        articles_per_month = [
            base_number
        ] * len(months)

        if extra_number > 0:

            extra_positions = [
                round(
                    i * (len(months) - 1)
                    / (extra_number - 1)
                )
                if extra_number > 1
                else len(months) // 2
                for i in range(extra_number)
            ]

            for position in extra_positions:
                articles_per_month[position] += 1

    # Сначала формируем сравнительную выборку
    # за предыдущие 24 месяца
    for month_number, month in enumerate(months):

        articles_to_take = articles_per_month[
            month_number
        ]

        if articles_to_take == 0:
            continue

        candidates = fetch_month_from_arxiv(
            category,
            month.year,
            month.month,
        )

        if len(candidates) < articles_to_take:
            raise ValueError(
                f"Недостаточно публикаций "
                f"в {category} за {month}"
            )

        selected_articles = random.sample(
            candidates,
            articles_to_take,
        )

        for article in selected_articles:
            article["period"] = "PREVIOUS"
            previous_rows.append(article)

        print(
            f"PREVIOUS {month}: "
            f"+{articles_to_take}, "
            f"всего {len(previous_rows)}/"
            f"{n_per_group}"
        )

        time.sleep(2)

    # После формирования сравнительной выборки
    # выбираем публикации анализируемого месяца
    recent_candidates = fetch_month_from_arxiv(
        category,
        recent_year,
        recent_month,
    )

    if len(recent_candidates) < n_per_group:
        raise ValueError(
            f"В {category} найдено только "
            f"{len(recent_candidates)} публикаций, "
            f"а требуется {n_per_group}"
        )

    recent_rows = random.sample(
        recent_candidates,
        n_per_group,
    )

    for article in recent_rows:
        article["period"] = "RECENT"

    # Объединяем обе части корпуса
    all_rows = previous_rows + recent_rows

    corpus_df = pd.DataFrame(all_rows)

    # Проверяем размер корпуса
    if len(corpus_df) != n_per_group * 2:
        raise ValueError(
            "Размер корпуса не соответствует ожидаемому"
        )

    # Проверяем отсутствие повторяющихся публикаций
    if corpus_df["arxiv_id"].nunique() != len(corpus_df):
        raise ValueError(
            "В корпусе обнаружены повторяющиеся arXiv ID"
        )

    print()
    print("Корпус сформирован.")
    print(corpus_df["period"].value_counts())

    return corpus_df

## 4. Получение полных текстов

Для каждой публикации сначала выполняется попытка получить HTML-версию статьи. Если HTML недоступен или содержит слишком мало текста, используется PDF.

PDF обрабатывается средствами `pypdf`. OCR в данном pipeline не используется. Источник текста сохраняется в поле `text_source`, что позволяет отличать статьи, полученные из HTML и PDF.


In [65]:
def get_text_from_html(arxiv_id):

    url = f"https://arxiv.org/html/{arxiv_id}"

    headers = {
        "User-Agent":
        "Mozilla/5.0 academic-research"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=90,
    )

    if response.status_code != 200:
        return None

    soup = BeautifulSoup(
        response.text,
        "html.parser",
    )

    unwanted_tags = soup([
        "script",
        "style",
        "nav",
    ])

    for tag in unwanted_tags:
        tag.decompose()

    text = soup.get_text(
        " ",
        strip=True,
    )

    if len(text) < 1000:
        return None

    return text


def get_text_from_pdf(arxiv_id):

    url = f"https://arxiv.org/pdf/{arxiv_id}"

    headers = {
        "User-Agent":
        "Mozilla/5.0 academic-research"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=120,
    )

    if response.status_code != 200:
        return None

    pdf_file = BytesIO(
        response.content
    )

    reader = PdfReader(pdf_file)

    pages_text = []

    for page in reader.pages:

        page_text = page.extract_text()

        if page_text is not None:
            pages_text.append(page_text)

    text = "\n".join(pages_text)

    if len(text) < 1000:
        return None

    return text

In [66]:
def download_corpus_texts(corpus_df):

    corpus_df = corpus_df.copy()

    corpus_df["full_text"] = None
    corpus_df["text_source"] = None

    total = len(corpus_df)

    for position, row_index in enumerate(
        corpus_df.index,
        start=1,
    ):

        arxiv_id = corpus_df.at[
            row_index,
            "arxiv_id",
        ]

        text = None
        source = None

        # Сначала HTML
        try:

            text = get_text_from_html(
                arxiv_id
            )

            if text is not None:
                source = "html"

        except Exception as error:

            print(
                "Ошибка HTML:",
                arxiv_id,
                error,
            )

        # Если HTML не сработал - PDF
        if text is None:

            try:

                text = get_text_from_pdf(
                    arxiv_id
                )

                if text is not None:
                    source = "pdf"

            except Exception as error:

                print(
                    "Ошибка PDF:",
                    arxiv_id,
                    error,
                )

        corpus_df.at[
            row_index,
            "full_text",
        ] = text

        corpus_df.at[
            row_index,
            "text_source",
        ] = source

        if text is None:
            text_length = 0
        else:
            text_length = len(text)

        print(
            f"{position}/{total} | "
            f"{arxiv_id} | "
            f"{source} | "
            f"{text_length:,} символов"
        )

        time.sleep(2)

    return corpus_df

## 5. Предобработка текста

Перед анализом текст переводится в нижний регистр. Удаляются URL, LaTeX-команды, отдельные числа и лишние символы. После этого нормализуются пробелы.

Исходный полный текст сохраняется отдельно в `full_text`, а очищенная версия для анализа - в `analysis_text`. Дополнительно проверяется, что полный текст удалось получить для каждой статьи корпуса.


In [67]:
LATEX_ARTIFACTS = [
    "displaystyle",
    "textstyle",
    "scriptstyle",
    "scriptscriptstyle",
    "begin",
    "end",
]


def clean_text(text):
    if text is None:
        return ""

    # Приводим весь текст к нижнему регистру
    cleaned = text.lower()

    # Удаляем технические слова LaTeX,
    # которые не несут содержательной информации
    for token in LATEX_ARTIFACTS:
        pattern = rf"\b{re.escape(token)}\b"
        cleaned = re.sub(
            pattern,
            " ",
            cleaned,
        )

    # Удаляем команды LaTeX
    cleaned = re.sub(
        r"\\[a-zA-Z]+",
        " ",
        cleaned,
    )

    # Удаляем отдельно стоящие числа
    cleaned = re.sub(
        r"\b\d+\b",
        " ",
        cleaned,
    )

    # Заменяем последовательности пробелов одним пробелом
    cleaned = re.sub(
        r"\s+",
        " ",
        cleaned,
    )

    return cleaned.strip()

In [68]:
def prepare_corpus(corpus_df):

    corpus_df = corpus_df.copy()

    corpus_df["text_length"] = (
        corpus_df["full_text"]
        .fillna("")
        .str.len()
    )

    missing_texts = (
        corpus_df["full_text"]
        .isna()
        .sum()
    )

    if missing_texts > 0:
        raise ValueError(
            f"Не удалось получить "
            f"{missing_texts} полных текстов"
        )

    corpus_df["analysis_text"] = (
        corpus_df["full_text"]
        .apply(clean_text)
    )

    print("Предобработка завершена.")

    return corpus_df

## 6. Анализ TF-IDF

Для каждой новой статьи создается отдельный набор документов, состоящий из 50 предыдущих публикаций и одной анализируемой новой статьи.

`TfidfVectorizer` рассматривает отдельные слова, биграммы и триграммы. В основной версии pipeline сохраняются термины с TF-IDF score выше 0.1. Этот порог был выбран после дополнительной проверки чувствительности результатов к разным значениям порога.


In [69]:
def run_tfidf(
    recent_df,
    previous_df,
    threshold=0.1,
):

    previous_texts = (
        previous_df["analysis_text"]
        .tolist()
    )

    tfidf_rows = []

    total = len(recent_df)

    for article_number, (_, article) in enumerate(
        recent_df.iterrows(),
        start=1,
    ):

        documents = previous_texts.copy()

        documents.append(
            article["analysis_text"]
        )

        vectorizer = TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 3),
            min_df=1,
            max_df=0.95,
        )

        tfidf_matrix = (
            vectorizer.fit_transform(
                documents
            )
        )

        terms = (
            vectorizer
            .get_feature_names_out()
        )

        recent_scores = (
            tfidf_matrix[-1]
            .toarray()[0]
        )

        found = 0

        for term_index in range(len(terms)):

            score = recent_scores[
                term_index
            ]

            if score > threshold:

                tfidf_rows.append({
                    "arxiv_id":
                        article["arxiv_id"],

                    "title":
                        article["title"],

                    "term":
                        terms[term_index],

                    "tfidf_score":
                        float(score),
                })

                found += 1

        print(
            f"TF-IDF {article_number}/{total} | "
            f"найдено > {threshold}: {found}"
        )

    tfidf_df = pd.DataFrame(
        tfidf_rows,
        columns=[
            "arxiv_id",
            "title",
            "term",
            "tfidf_score",
        ],
    )

    return tfidf_df

## 7. Анализ KeyBERT

KeyBERT применяется отдельно к каждой новой публикации. Модель `all-MiniLM-L6-v2` оценивает, насколько слово или словосочетание связано с содержанием полного текста.

Для дальнейшего анализа сохраняются кандидаты со score выше 0.4. Параметр `top_n` задает максимальное число кандидатов, рассматриваемых для одной статьи. В финальной конфигурации используется `top_n=2000`.

Pipeline также контролирует минимальный score среди полученных кандидатов. Если все 2000 кандидатов имеют score выше 0.4, выводится информация о том, что `top_n` потенциально может ограничивать результат. Это предупреждение не означает ошибку в статье или в pipeline - оно показывает, что для данной статьи могут существовать дополнительные кандидаты выше выбранного порога.


In [70]:
def run_keybert(
    recent_df,
    threshold=0.4,
    top_n=2000,
):

    print("Загружаем модель KeyBERT...")

    kw_model = KeyBERT(
        model="all-MiniLM-L6-v2"
    )

    keybert_rows = []

    total = len(recent_df)

    articles_at_limit = 0

    for article_number, (_, article) in enumerate(
        recent_df.iterrows(),
        start=1,
    ):

        keywords = kw_model.extract_keywords(
            article["analysis_text"],
            keyphrase_ngram_range=(1, 3),
            stop_words="english",
            top_n=top_n,
        )

        found = 0

        for phrase, score in keywords:

            if score > threshold:

                keybert_rows.append({
                    "arxiv_id":
                        article["arxiv_id"],

                    "title":
                        article["title"],

                    "keyword":
                        phrase,

                    "keybert_score":
                        float(score),
                })

                found += 1

        if len(keywords) > 0:

            lowest_score = keywords[-1][1]

            if (
                len(keywords) == top_n
                and lowest_score > threshold
            ):
                articles_at_limit += 1

            print(
                f"KeyBERT "
                f"{article_number}/{total} | "
                f"найдено > {threshold}: "
                f"{found} | "
                f"min score: "
                f"{lowest_score:.3f}"
            )

    keybert_df = pd.DataFrame(
        keybert_rows,
        columns=[
            "arxiv_id",
            "title",
            "keyword",
            "keybert_score",
        ],
    )

    print()
    print(
        "Статей, для которых top_n "
        "может ограничивать результат:",
        articles_at_limit,
    )

    return keybert_df, articles_at_limit

## 8. Сопоставление TF-IDF и KeyBERT

Результаты двух методов сопоставляются отдельно для каждой новой статьи. Перед сравнением термины приводятся к нижнему регистру и нормализуются пробелы.

В пересечение попадают только точные совпадения нормализованных слов и словосочетаний внутри одной и той же публикации. Семантически близкие, но записанные по-разному выражения не считаются совпадениями.


In [71]:
def normalize_term(term):

    term = str(term).lower()

    term = re.sub(
        r"\s+",
        " ",
        term,
    )

    return term.strip()


def compare_methods(
    tfidf_df,
    keybert_df,
):

    tfidf_for_merge = tfidf_df.copy()
    keybert_for_merge = keybert_df.copy()

    tfidf_for_merge[
        "normalized_term"
    ] = (
        tfidf_for_merge["term"]
        .apply(normalize_term)
    )

    keybert_for_merge[
        "normalized_term"
    ] = (
        keybert_for_merge["keyword"]
        .apply(normalize_term)
    )

    intersection_df = pd.merge(
        tfidf_for_merge,
        keybert_for_merge,
        on=[
            "arxiv_id",
            "normalized_term",
        ],
        how="inner",
        suffixes=(
            "_tfidf",
            "_keybert",
        ),
    )

    return intersection_df

## 9. Формирование и сохранение JSON

Pipeline создает два файла для каждой категории.

`corpus.json` содержит метаданные, полный текст и очищенный текст каждой публикации.

`results.json` содержит параметры запуска, сводные показатели и результаты TF-IDF, KeyBERT и их пересечения для каждой новой статьи.

Параметры анализа сохраняются вместе с результатами, чтобы было понятно, при каких настройках был получен конкретный результат.


In [72]:
def create_json_result(
    category,
    recent_year,
    recent_month,
    corpus_df,
    tfidf_df,
    keybert_df,
    intersection_df,
    tfidf_threshold,
    keybert_threshold,
    keybert_top_n,
    articles_at_limit,
):

    result = {

        "metadata": {
            "source": "arXiv",
            "category": category,
            "recent_year": recent_year,
            "recent_month": recent_month,
            "number_of_articles":
                len(corpus_df),
            "number_recent":
                int(
                    (
                        corpus_df["period"]
                        == "RECENT"
                    ).sum()
                ),
            "number_previous":
                int(
                    (
                        corpus_df["period"]
                        == "PREVIOUS"
                    ).sum()
                ),
        },

        "parameters": {
            "tfidf_threshold":
                tfidf_threshold,

            "keybert_threshold":
                keybert_threshold,

            "keybert_top_n":
                keybert_top_n,

            "ngram_min": 1,
            "ngram_max": 3,

            "random_seed":
                RANDOM_SEED,
        },

        "summary": {
            "unique_tfidf_terms":
                int(
                    tfidf_df["term"]
                    .nunique()
                ),

            "unique_keybert_terms":
                int(
                    keybert_df["keyword"]
                    .nunique()
                ),

            "unique_intersection_terms":
                int(
                    intersection_df[
                        "normalized_term"
                    ].nunique()
                ),

            "articles_with_intersection":
                int(
                    intersection_df[
                        "arxiv_id"
                    ].nunique()
                ),

            "articles_limited_by_top_n":
                int(articles_at_limit),
        },

        "articles": [],
    }

    recent_df = corpus_df[
        corpus_df["period"] == "RECENT"
    ]

    for _, article in recent_df.iterrows():

        arxiv_id = article["arxiv_id"]

        article_tfidf = tfidf_df[
            tfidf_df["arxiv_id"]
            == arxiv_id
        ]

        article_keybert = keybert_df[
            keybert_df["arxiv_id"]
            == arxiv_id
        ]

        article_intersection = (
            intersection_df[
                intersection_df["arxiv_id"]
                == arxiv_id
            ]
        )

        article_result = {

            "arxiv_id":
                arxiv_id,

            "title":
                article["title"],

            "category":
                category,

            "month":
                article["month"],

            "text_source":
                article["text_source"],

            "text_length":
                int(article["text_length"]),

            "tfidf_terms": [],

            "keybert_terms": [],

            "intersection": [],
        }

        for _, row in article_tfidf.iterrows():

            article_result[
                "tfidf_terms"
            ].append({
                "term":
                    row["term"],

                "score":
                    float(
                        row["tfidf_score"]
                    ),
            })

        for _, row in article_keybert.iterrows():

            article_result[
                "keybert_terms"
            ].append({
                "term":
                    row["keyword"],

                "score":
                    float(
                        row["keybert_score"]
                    ),
            })

        article_result[
            "intersection"
        ] = (
            article_intersection[
                "normalized_term"
            ]
            .drop_duplicates()
            .tolist()
        )

        result["articles"].append(
            article_result
        )

    return result

In [73]:
def save_json(
    data,
    file_path,
):

    with open(
        file_path,
        "w",
        encoding="utf-8",
        errors="replace",
    ) as file:

        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2,
        )

    print(
        "JSON сохранён:",
        file_path,
    )

In [74]:
def save_corpus_json(
    corpus_df,
    file_path,
):

    corpus_records = []

    for _, article in corpus_df.iterrows():

        record = {
            "arxiv_id":
                article["arxiv_id"],

            "title":
                article["title"],

            "subjects":
                article["subjects"],

            "category":
                article["category"],

            "month":
                article["month"],

            "period":
                article["period"],

            "text_source":
                article["text_source"],

            "full_text":
                article["full_text"],

            "analysis_text":
                article["analysis_text"],
        }

        corpus_records.append(record)

    with open(
        file_path,
        "w",
        encoding="utf-8",
        errors="replace",
    ) as file:

        json.dump(
            corpus_records,
            file,
            ensure_ascii=False,
            indent=2,
        )

    print(
        "Корпус сохранён:",
        file_path,
    )

## 10. Проверка выходных файлов

После сохранения JSON выполняется автоматическая проверка. Проверяется количество статей, наличие обязательных полей, наличие полных текстов, структура файла результатов и количество новых публикаций.

Сообщения проверки сформулированы текстом без специальных символов, чтобы вывод одинаково читался в разных средах.


In [75]:
def validate_json_files(
    corpus_file,
    results_file,
    expected_n_per_group,
):

    # Загружаем corpus JSON для проверки
    with open(
        corpus_file,
        "r",
        encoding="utf-8",
    ) as file:
        corpus_data = json.load(file)

    # Загружаем results JSON для проверки
    with open(
        results_file,
        "r",
        encoding="utf-8",
    ) as file:
        results_data = json.load(file)

    print()
    print("Проверка JSON")

    # Проверяем количество публикаций в корпусе
    expected_total = expected_n_per_group * 2

    if len(corpus_data) == expected_total:
        print(
            f"Количество статей в corpus.json корректно: "
            f"{len(corpus_data)}"
        )
    else:
        print(
            f"Ошибка количества статей в corpus.json. "
            f"Ожидалось {expected_total}, получено {len(corpus_data)}"
        )

    # Проверяем обязательные поля каждой записи корпуса
    corpus_fields = [
        "arxiv_id",
        "title",
        "subjects",
        "category",
        "month",
        "period",
        "text_source",
        "full_text",
        "analysis_text",
    ]

    missing_corpus_fields = []

    for article_number, article in enumerate(
        corpus_data,
        start=1,
    ):
        for field in corpus_fields:
            if field not in article:
                missing_corpus_fields.append(
                    f"article {article_number}: {field}"
                )

    if len(missing_corpus_fields) == 0:
        print("Все обязательные поля corpus.json присутствуют")
    else:
        print(
            "В corpus.json отсутствуют поля:",
            missing_corpus_fields,
        )

    # Проверяем, что у каждой статьи сохранен полный текст
    empty_texts = 0

    for article in corpus_data:
        if not article.get("full_text"):
            empty_texts += 1

    if empty_texts == 0:
        print("Полные тексты присутствуют у всех статей")
    else:
        print(
            f"Статей без полного текста: {empty_texts}"
        )

    # Проверяем основные разделы results JSON
    result_fields = [
        "metadata",
        "parameters",
        "summary",
        "articles",
    ]

    missing_result_fields = []

    for field in result_fields:
        if field not in results_data:
            missing_result_fields.append(field)

    if len(missing_result_fields) == 0:
        print("Структура results.json корректна")
    else:
        print(
            "В results.json отсутствуют разделы:",
            missing_result_fields,
        )

    # Проверяем количество новых публикаций в results JSON
    recent_count = len(
        results_data.get("articles", [])
    )

    if recent_count == expected_n_per_group:
        print(
            "Количество новых статей в results.json корректно"
        )
    else:
        print(
            f"Количество новых статей в results.json некорректно: "
            f"{recent_count}"
        )

    print("Проверка JSON завершена")


## 11. Повторное использование уже сохраненного корпуса

Наиболее медленная часть повторного запуска - повторное обращение к arXiv и загрузка 100 полных текстов. Поэтому при повторном анализе можно использовать уже сохраненный `corpus.json`.

Функция ниже загружает сохраненный корпус только в том случае, если категория, месяц новой выборки и количество публикаций соответствуют текущему запуску. Если файл не подходит, pipeline автоматически формирует корпус заново.

Это ускоряет повторные расчеты и одновременно сохраняет тот же набор статей, что особенно важно при сравнении разных параметров анализа.


In [76]:
def load_saved_corpus(
    corpus_file,
    category,
    recent_year,
    recent_month,
    n_per_group,
):

    if not corpus_file.exists():
        return None

    try:
        with open(
            corpus_file,
            "r",
            encoding="utf-8",
        ) as file:
            records = json.load(file)

        corpus_df = pd.DataFrame(records)

        expected_total = n_per_group * 2
        expected_month = (
            f"{recent_year}-{recent_month:02d}"
        )

        if len(corpus_df) != expected_total:
            return None

        if (
            corpus_df["category"]
            .ne(category)
            .any()
        ):
            return None

        recent_df = corpus_df[
            corpus_df["period"] == "RECENT"
        ]

        previous_df = corpus_df[
            corpus_df["period"] == "PREVIOUS"
        ]

        if len(recent_df) != n_per_group:
            return None

        if len(previous_df) != n_per_group:
            return None

        if (
            recent_df["month"]
            .ne(expected_month)
            .any()
        ):
            return None

        # text_length нужен для results JSON,
        # поэтому восстанавливаем его из полного текста
        corpus_df["text_length"] = (
            corpus_df["full_text"]
            .fillna("")
            .str.len()
        )

        print(
            "Используется ранее сохраненный корпус:",
            corpus_file,
        )

        return corpus_df

    except (
        OSError,
        ValueError,
        KeyError,
        json.JSONDecodeError,
    ):
        return None


## 12. Основная функция pipeline

`run_pipeline()` объединяет все этапы в один последовательный процесс.

Параметр `reuse_saved_corpus=True` позволяет при повторном запуске использовать уже сохраненный корпус и не скачивать те же статьи заново. При первом запуске или при отсутствии подходящего файла корпус будет сформирован автоматически.

Для финального анализа используются следующие основные параметры:

- 50 новых и 50 предыдущих публикаций
- TF-IDF threshold 0.1
- KeyBERT threshold 0.4
- KeyBERT top_n 2000
- n-граммы длиной от 1 до 3 слов


In [77]:
def run_pipeline(
    category,
    recent_year=2026,
    recent_month=7,
    n_per_group=50,
    tfidf_threshold=0.1,
    keybert_threshold=0.4,
    keybert_top_n=2000,
    reuse_saved_corpus=True,
):

    print()
    print("ARXIV PHYSICS PIPELINE")
    print("Категория:", category)

    corpus_file = (
        DATA_DIR /
        f"{category}_corpus.json"
    )

    results_file = (
        DATA_DIR /
        f"{category}_results.json"
    )

    corpus_df = None

    # При повторном запуске сначала пытаемся
    # использовать уже сохраненный корпус
    if reuse_saved_corpus:
        corpus_df = load_saved_corpus(
            corpus_file=corpus_file,
            category=category,
            recent_year=recent_year,
            recent_month=recent_month,
            n_per_group=n_per_group,
        )

    # Если подходящего сохраненного корпуса нет,
    # формируем его и загружаем полные тексты
    if corpus_df is None:

        print()
        print("1. Формирование корпуса")

        corpus_df = build_corpus(
            category=category,
            recent_year=recent_year,
            recent_month=recent_month,
            n_per_group=n_per_group,
        )

        print()
        print("2. Получение полных текстов")

        corpus_df = download_corpus_texts(
            corpus_df
        )

        print()
        print("3. Предобработка")

        corpus_df = prepare_corpus(
            corpus_df
        )

    else:
        print()
        print(
            "Этапы формирования корпуса и загрузки "
            "полных текстов пропущены"
        )

    # Разделяем корпус на предыдущие и новые публикации
    previous_df = corpus_df[
        corpus_df["period"] == "PREVIOUS"
    ].copy()

    recent_df = corpus_df[
        corpus_df["period"] == "RECENT"
    ].copy()

    print()
    print("4. TF-IDF")

    tfidf_df = run_tfidf(
        recent_df=recent_df,
        previous_df=previous_df,
        threshold=tfidf_threshold,
    )

    print()
    print("5. KeyBERT")

    keybert_df, articles_at_limit = (
        run_keybert(
            recent_df=recent_df,
            threshold=keybert_threshold,
            top_n=keybert_top_n,
        )
    )

    print()
    print("6. Сопоставление методов")

    intersection_df = compare_methods(
        tfidf_df,
        keybert_df,
    )

    result_json = create_json_result(
        category=category,
        recent_year=recent_year,
        recent_month=recent_month,
        corpus_df=corpus_df,
        tfidf_df=tfidf_df,
        keybert_df=keybert_df,
        intersection_df=intersection_df,
        tfidf_threshold=tfidf_threshold,
        keybert_threshold=keybert_threshold,
        keybert_top_n=keybert_top_n,
        articles_at_limit=articles_at_limit,
    )

    print()
    print("7. Сохранение результатов")

    save_corpus_json(
        corpus_df,
        corpus_file,
    )

    save_json(
        result_json,
        results_file,
    )

    validate_json_files(
        corpus_file=corpus_file,
        results_file=results_file,
        expected_n_per_group=n_per_group,
    )

    print()
    print("Pipeline завершен")

    print(
        "Уникальных TF-IDF терминов:",
        tfidf_df["term"].nunique(),
    )

    print(
        "Уникальных KeyBERT терминов:",
        keybert_df["keyword"].nunique(),
    )

    print(
        "Уникальных совпавших терминов:",
        intersection_df[
            "normalized_term"
        ].nunique(),
    )

    print(
        "Статей с совпадением:",
        intersection_df[
            "arxiv_id"
        ].nunique(),
    )

    return {
        "corpus": corpus_df,
        "tfidf": tfidf_df,
        "keybert": keybert_df,
        "intersection": intersection_df,
        "json": result_json,
    }


# Полные запуски по категориям

Ниже выполняется одна и та же финальная версия pipeline для трех разных областей физики. Это позволяет проверить, что методика не привязана к одной конкретной категории arXiv.

Повторные тестовые запуски на 3 статьях и дублирующий запуск `quant-ph` удалены из финальной версии notebook. Они были полезны на этапе разработки, но в итоговом коде только увеличивали время выполнения и могли перезаписывать результаты.


## 13. Quantum Physics

Категория `quant-ph` использовалась для первоначальной разработки и проверки методики. Полный запуск формирует корпус из 50 публикаций за июль 2026 года и 50 публикаций за предыдущие два года.

При последнем полном запуске до финальной очистки notebook было получено 591 уникальный TF-IDF термин, 14279 уникальных KeyBERT терминов, 51 уникальный термин в пересечении и 20 статей с хотя бы одним точным совпадением.

Для всех 50 статей минимальный KeyBERT score среди рассмотренных кандидатов опустился ниже порога 0.4, поэтому `top_n=2000` не ограничивал этот запуск.


## Итоговый анализ категории quant-ph

На этом этапе универсальный pipeline применяется к категории квантовой физики `quant-ph`.

Корпус включает 100 публикаций arXiv:

- 50 публикаций за июль 2026 года
- 50 публикаций за предыдущие 24 месяца
- сравнительная выборка формируется приблизительно равномерно по двухлетнему периоду

Для каждой публикации сохраняется полный текст. Перед анализом выполняется единая предобработка текста.

Для выделения характерных слов и словосочетаний используются два метода:

- TF-IDF с n-граммами длиной от 1 до 3 слов
- KeyBERT с n-граммами длиной от 1 до 3 слов

Каждая новая публикация при расчёте TF-IDF рассматривается относительно общего корпуса из 50 публикаций предыдущего периода.

Первоначально для TF-IDF был рассмотрен строгий порог 0.6. Анализ чувствительности показал, что при таком значении отбирается слишком мало терминов, поэтому для расширенного анализа используется порог 0.1.

Для KeyBERT сохраняются термины со значением выше 0.4.

После выполнения двух методов проводится точное сопоставление нормализованных терминов в пределах одной публикации. Итоговый результат показывает количество терминов, обнаруженных одновременно TF-IDF и KeyBERT, а также количество публикаций, в которых были найдены такие совпадения.

In [78]:
quant_results = run_pipeline(
    category="quant-ph",
    recent_year=2026,
    recent_month=7,
    n_per_group=50,
    tfidf_threshold=0.1,
    keybert_threshold=0.4,
    keybert_top_n=2000,
    reuse_saved_corpus=True,
)



ARXIV PHYSICS PIPELINE
Категория: quant-ph
Используется ранее сохраненный корпус: pipeline_data\quant-ph_corpus.json

Этапы формирования корпуса и загрузки полных текстов пропущены

4. TF-IDF
TF-IDF 1/50 | найдено > 0.1: 13
TF-IDF 2/50 | найдено > 0.1: 14
TF-IDF 3/50 | найдено > 0.1: 16
TF-IDF 4/50 | найдено > 0.1: 12
TF-IDF 5/50 | найдено > 0.1: 14
TF-IDF 6/50 | найдено > 0.1: 12
TF-IDF 7/50 | найдено > 0.1: 13
TF-IDF 8/50 | найдено > 0.1: 15
TF-IDF 9/50 | найдено > 0.1: 15
TF-IDF 10/50 | найдено > 0.1: 11
TF-IDF 11/50 | найдено > 0.1: 12
TF-IDF 12/50 | найдено > 0.1: 12
TF-IDF 13/50 | найдено > 0.1: 12
TF-IDF 14/50 | найдено > 0.1: 19
TF-IDF 15/50 | найдено > 0.1: 11
TF-IDF 16/50 | найдено > 0.1: 11
TF-IDF 17/50 | найдено > 0.1: 13
TF-IDF 18/50 | найдено > 0.1: 17
TF-IDF 19/50 | найдено > 0.1: 17
TF-IDF 20/50 | найдено > 0.1: 16
TF-IDF 21/50 | найдено > 0.1: 15
TF-IDF 22/50 | найдено > 0.1: 12
TF-IDF 23/50 | найдено > 0.1: 19
TF-IDF 24/50 | найдено > 0.1: 13
TF-IDF 25/50 | найдено >

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9077.24it/s]


KeyBERT 1/50 | найдено > 0.4: 41 | min score: 0.144
KeyBERT 2/50 | найдено > 0.4: 820 | min score: 0.286
KeyBERT 3/50 | найдено > 0.4: 303 | min score: 0.183
KeyBERT 4/50 | найдено > 0.4: 199 | min score: 0.192
KeyBERT 5/50 | найдено > 0.4: 245 | min score: 0.210
KeyBERT 6/50 | найдено > 0.4: 239 | min score: 0.174
KeyBERT 7/50 | найдено > 0.4: 152 | min score: 0.115
KeyBERT 8/50 | найдено > 0.4: 260 | min score: 0.189
KeyBERT 9/50 | найдено > 0.4: 302 | min score: 0.116
KeyBERT 10/50 | найдено > 0.4: 413 | min score: 0.142
KeyBERT 11/50 | найдено > 0.4: 215 | min score: 0.183
KeyBERT 12/50 | найдено > 0.4: 254 | min score: 0.168
KeyBERT 13/50 | найдено > 0.4: 209 | min score: 0.159
KeyBERT 14/50 | найдено > 0.4: 323 | min score: 0.286
KeyBERT 15/50 | найдено > 0.4: 181 | min score: 0.146
KeyBERT 16/50 | найдено > 0.4: 177 | min score: 0.016
KeyBERT 17/50 | найдено > 0.4: 725 | min score: 0.278
KeyBERT 18/50 | найдено > 0.4: 162 | min score: 0.182
KeyBERT 19/50 | найдено > 0.4: 262 | m

### Проверка состава корпуса Quantum Physics

Следующая ячейка выводит arXiv ID всех публикаций, вошедших в две группы. Этот список полезен для проверки воспроизводимости и позволяет убедиться, что при сравнении разных запусков используется один и тот же корпус.


In [79]:
print("RECENT:")
print(
    quant_results["corpus"]
    .query("period == 'RECENT'")["arxiv_id"]
    .tolist()
)

print("\nPREVIOUS:")
print(
    quant_results["corpus"]
    .query("period == 'PREVIOUS'")["arxiv_id"]
    .tolist()
)

RECENT:
['2607.26093', '2607.04586', '2607.01035', '2607.01620', '2607.11410', '2607.09912', '2607.08977', '2607.05897', '2607.00994', '2607.04110', '2607.27342', '2607.01446', '2607.22294', '2607.21995', '2607.03527', '2607.24299', '2607.16849', '2607.01323', '2607.01217', '2607.03897', '2607.08767', '2607.09381', '2607.20605', '2607.24686', '2607.01085', '2607.22995', '2607.08166', '2607.29012', '2607.26474', '2607.28493', '2607.16765', '2607.08882', '2607.18050', '2607.24258', '2607.11516', '2607.10847', '2607.19166', '2607.00266', '2607.04766', '2607.10415', '2607.06847', '2607.28383', '2607.16857', '2607.13722', '2607.06675', '2607.08697', '2607.05190', '2607.13521', '2607.03805', '2607.15201']

PREVIOUS:
['2407.05019', '2407.18474', '2407.17732', '2408.11671', '2408.01593', '2409.08816', '2409.19683', '2410.06429', '2410.19770', '2411.03734', '2411.14838', '2412.14753', '2412.17781', '2501.10291', '2501.04526', '2502.02698', '2502.13393', '2503.14598', '2503.04620', '2504.10771',

## 14. Astrophysics

Вторая полная проверка выполняется для категории `astro-ph`. Она показывает, что тот же pipeline может применяться к другой области физики без изменения функций анализа.

При последнем полном запуске было получено 491 уникальный TF-IDF термин, 23119 уникальных KeyBERT терминов, 38 уникальных терминов в пересечении и 28 статей с хотя бы одним точным совпадением.

Для этой категории страницы arXiv в некоторых месяцах достигали лимита 2000 публикаций. Pipeline сообщает об этом отдельно, поскольку исходный месячный список потенциально может быть неполным.


In [80]:
astro_results = run_pipeline(
    category="astro-ph",
    recent_year=2026,
    recent_month=7,
    n_per_group=50,
    tfidf_threshold=0.1,
    keybert_threshold=0.4,
    keybert_top_n=2000,
    reuse_saved_corpus=True,
)



ARXIV PHYSICS PIPELINE
Категория: astro-ph
Используется ранее сохраненный корпус: pipeline_data\astro-ph_corpus.json

Этапы формирования корпуса и загрузки полных текстов пропущены

4. TF-IDF
TF-IDF 1/50 | найдено > 0.1: 10
TF-IDF 2/50 | найдено > 0.1: 17
TF-IDF 3/50 | найдено > 0.1: 14
TF-IDF 4/50 | найдено > 0.1: 11
TF-IDF 5/50 | найдено > 0.1: 9
TF-IDF 6/50 | найдено > 0.1: 22
TF-IDF 7/50 | найдено > 0.1: 20
TF-IDF 8/50 | найдено > 0.1: 17
TF-IDF 9/50 | найдено > 0.1: 19
TF-IDF 10/50 | найдено > 0.1: 20
TF-IDF 11/50 | найдено > 0.1: 18
TF-IDF 12/50 | найдено > 0.1: 10
TF-IDF 13/50 | найдено > 0.1: 13
TF-IDF 14/50 | найдено > 0.1: 20
TF-IDF 15/50 | найдено > 0.1: 15
TF-IDF 16/50 | найдено > 0.1: 19
TF-IDF 17/50 | найдено > 0.1: 22
TF-IDF 18/50 | найдено > 0.1: 25
TF-IDF 19/50 | найдено > 0.1: 26
TF-IDF 20/50 | найдено > 0.1: 6
TF-IDF 21/50 | найдено > 0.1: 9
TF-IDF 22/50 | найдено > 0.1: 17
TF-IDF 23/50 | найдено > 0.1: 23
TF-IDF 24/50 | найдено > 0.1: 20
TF-IDF 25/50 | найдено > 0.

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6639.11it/s]


KeyBERT 1/50 | найдено > 0.4: 722 | min score: 0.249
KeyBERT 2/50 | найдено > 0.4: 252 | min score: 0.149
KeyBERT 3/50 | найдено > 0.4: 323 | min score: 0.263
KeyBERT 4/50 | найдено > 0.4: 453 | min score: 0.235
KeyBERT 5/50 | найдено > 0.4: 60 | min score: 0.160
KeyBERT 6/50 | найдено > 0.4: 985 | min score: 0.311
KeyBERT 7/50 | найдено > 0.4: 1157 | min score: 0.347
KeyBERT 8/50 | найдено > 0.4: 467 | min score: 0.216
KeyBERT 9/50 | найдено > 0.4: 463 | min score: 0.281
KeyBERT 10/50 | найдено > 0.4: 1310 | min score: 0.333
KeyBERT 11/50 | найдено > 0.4: 412 | min score: 0.203
KeyBERT 12/50 | найдено > 0.4: 460 | min score: 0.257
KeyBERT 13/50 | найдено > 0.4: 227 | min score: 0.198
KeyBERT 14/50 | найдено > 0.4: 152 | min score: 0.210
KeyBERT 15/50 | найдено > 0.4: 561 | min score: 0.284
KeyBERT 16/50 | найдено > 0.4: 537 | min score: 0.257
KeyBERT 17/50 | найдено > 0.4: 89 | min score: 0.135
KeyBERT 18/50 | найдено > 0.4: 404 | min score: 0.241
KeyBERT 19/50 | найдено > 0.4: 926 | 

## 15. Atmospheric and Oceanic Physics

Третья проверка выполняется для категории `physics.ao-ph`. Эта область отличается от квантовой физики и астрофизики, поэтому ее использование дополнительно проверяет переносимость pipeline.

При последнем полном запуске было получено 550 уникальных TF-IDF терминов, 19381 уникальный KeyBERT термин, 53 уникальных термина в пересечении и 23 статьи с хотя бы одним точным совпадением.

Для одной статьи все 2000 рассмотренных KeyBERT кандидатов имели score выше 0.4. Поэтому для этой статьи `top_n=2000` потенциально ограничивает число найденных кандидатов. Это диагностическое предупреждение, а не ошибка выполнения. В итоговых данных оно сохраняется в поле `articles_limited_by_top_n`.


In [81]:
atmospheric_results = run_pipeline(
    category="physics.ao-ph",
    recent_year=2026,
    recent_month=7,
    n_per_group=50,
    tfidf_threshold=0.1,
    keybert_threshold=0.4,
    keybert_top_n=2000,
    reuse_saved_corpus=True,
)



ARXIV PHYSICS PIPELINE
Категория: physics.ao-ph
Используется ранее сохраненный корпус: pipeline_data\physics.ao-ph_corpus.json

Этапы формирования корпуса и загрузки полных текстов пропущены

4. TF-IDF
TF-IDF 1/50 | найдено > 0.1: 13
TF-IDF 2/50 | найдено > 0.1: 13
TF-IDF 3/50 | найдено > 0.1: 14
TF-IDF 4/50 | найдено > 0.1: 8
TF-IDF 5/50 | найдено > 0.1: 16
TF-IDF 6/50 | найдено > 0.1: 9
TF-IDF 7/50 | найдено > 0.1: 17
TF-IDF 8/50 | найдено > 0.1: 10
TF-IDF 9/50 | найдено > 0.1: 11
TF-IDF 10/50 | найдено > 0.1: 14
TF-IDF 11/50 | найдено > 0.1: 9
TF-IDF 12/50 | найдено > 0.1: 16
TF-IDF 13/50 | найдено > 0.1: 16
TF-IDF 14/50 | найдено > 0.1: 24
TF-IDF 15/50 | найдено > 0.1: 8
TF-IDF 16/50 | найдено > 0.1: 19
TF-IDF 17/50 | найдено > 0.1: 11
TF-IDF 18/50 | найдено > 0.1: 18
TF-IDF 19/50 | найдено > 0.1: 9
TF-IDF 20/50 | найдено > 0.1: 14
TF-IDF 21/50 | найдено > 0.1: 16
TF-IDF 22/50 | найдено > 0.1: 8
TF-IDF 23/50 | найдено > 0.1: 6
TF-IDF 24/50 | найдено > 0.1: 13
TF-IDF 25/50 | найден

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6626.38it/s]


KeyBERT 1/50 | найдено > 0.4: 212 | min score: 0.138
KeyBERT 2/50 | найдено > 0.4: 520 | min score: 0.280
KeyBERT 3/50 | найдено > 0.4: 799 | min score: 0.272
KeyBERT 4/50 | найдено > 0.4: 801 | min score: 0.228
KeyBERT 5/50 | найдено > 0.4: 1321 | min score: 0.301
KeyBERT 6/50 | найдено > 0.4: 202 | min score: 0.047
KeyBERT 7/50 | найдено > 0.4: 852 | min score: 0.232
KeyBERT 8/50 | найдено > 0.4: 164 | min score: 0.153
KeyBERT 9/50 | найдено > 0.4: 185 | min score: 0.186
KeyBERT 10/50 | найдено > 0.4: 69 | min score: 0.134
KeyBERT 11/50 | найдено > 0.4: 634 | min score: 0.194
KeyBERT 12/50 | найдено > 0.4: 279 | min score: 0.202
KeyBERT 13/50 | найдено > 0.4: 428 | min score: 0.232
KeyBERT 14/50 | найдено > 0.4: 353 | min score: 0.152
KeyBERT 15/50 | найдено > 0.4: 327 | min score: 0.224
KeyBERT 16/50 | найдено > 0.4: 333 | min score: 0.149
KeyBERT 17/50 | найдено > 0.4: 777 | min score: 0.182
KeyBERT 18/50 | найдено > 0.4: 394 | min score: 0.218
KeyBERT 19/50 | найдено > 0.4: 343 | 

# Итог

Разработан универсальный pipeline для автоматизированного сбора и анализа полнотекстовых научных публикаций arXiv по различным категориям физики.

Pipeline включает формирование корпуса из новых публикаций и публикаций предыдущего периода, получение полных текстов, предобработку, извлечение терминов и словосочетаний методами TF-IDF и KeyBERT, а также сопоставление результатов двух методов по точному совпадению нормализованных терминов внутри одной публикации.

Для анализа используются униграммы, биграммы и триграммы. В итоговой конфигурации применяется порог TF-IDF > 0.1 и порог KeyBERT score > 0.4. Параметр `top_n=2000` задаёт максимальное количество кандидатов KeyBERT, а дополнительная диагностическая проверка позволяет выявлять публикации, для которых это ограничение потенциально может влиять на результат.

Pipeline был проверен на категориях `quant-ph`, `astro-ph` и `physics.ao-ph`, что подтверждает возможность применения одной и той же процедуры к различным тематическим областям физики без изменения основной логики программы.

Для каждой категории сохраняются два JSON-файла:

- `*_corpus.json` - корпус из 100 публикаций с метаданными, полными и предобработанными текстами
- `*_results.json` - параметры анализа, сводные показатели и результаты TF-IDF, KeyBERT и их пересечения для 50 новых публикаций

Перед завершением работы автоматически проверяются количество публикаций, наличие обязательных полей, наличие полных текстов и структура итоговых JSON-файлов.

Таким образом, получен воспроизводимый и параметризованный инструмент, который позволяет сформировать корпус публикаций arXiv для выбранной категории физики, выполнить анализ двумя методами и сохранить как исходные данные, так и результаты в структурированном формате.